# Badge 2 : Collaboration, Marketplace & Cost Estimation Workshop

## Lesson 2 : Understanding the inbound shares


In [ ]:
%%sql -r dataframe_1
use role accountadmin;
alter database DB_SHARED_FROM_SNOWFLAKE rename to snowflake_sample_data

Attribuer des privilèges à d’autres rôles
Snowflake fournit un ensemble de privilèges pour travailler avec des annonces dans Snowflake Marketplace ou un Data Exchange.

Accord de privilèges d’administrateur dans un Data Exchange
Par défaut, seul un administrateur de compte (un utilisateur ayant le rôle ACCOUNTADMIN) dans le compte d’administrateur Data Exchange peut gérer un Data Exchange qui comprend les tâches suivantes :

- Gérer ou supprimer des membres.

- Approuver ou refuser les demandes d’approbation des annonces.

- Approuver ou refuser les demandes d’approbation de profils de fournisseurs.

- Afficher les catégories.

Pour prendre en charge la délégation de ces tâches à d’autres utilisateurs, le privilège **IMPORTED PRIVILEGES** peut être accordé sur un Data Exchange à d’autres rôles.

Attribution du privilège IMPORTED PRIVILEGES à d’autres rôles
Pour accorder le privilège IMPORTED PRIVILEGES sur un Data Exchange à un rôle, utilisez le rôle ACCOUNTADMIN et la commande GRANT <privilèges> … TO ROLE.

**NOTE :** 

Le paramètre WITH GRANT OPTION ne prend pas en charge le privilège IMPORTED PRIVILEGES.

**SYNTAXE :**

Copy
GRANT IMPORTED PRIVILEGES ON DATA EXCHANGE <exchange_name> TO <role_name>;

### 🥋 Use Select Statements to Look at Sample Data

In [ ]:
%%sql -r dataframe_2
use role sysadmin;
--Check the range of values in the Market Segment Column
select distinct c_mktsegment
from snowflake_sample_data.tpch_sf1.customer;

In [ ]:
%%sql -r dataframe_3
--Find out which Market Segments have the most customers
select c_mktsegment, count(*)
from snowflake_sample_data.tpch_sf1.customer
group by c_mktsegment
order by count(*);

### 🥋 Join and Aggregate Shared Data

In [ ]:
%%sql -r dataframe_4
-- Nations Table
select n_nationkey, n_name, n_regionkey
from snowflake_sample_data.tpch_sf1.nation;

In [ ]:
%%sql -r dataframe_5
-- Join the Tables and Sort
select r_name as region, n_name as nation
from snowflake_sample_data.tpch_sf1.nation
join snowflake_sample_data.tpch_sf1.region
on n_regionkey = r_regionkey
order by r_name, n_name asc;

In [ ]:
%%sql -r dataframe_6
--Group and Count Rows Per Region
select r_name as region, count(n_name) as num_countries
from snowflake_sample_data.tpch_sf1.nation
join snowflake_sample_data.tpch_sf1.region
on n_regionkey = r_regionkey
group by r_name;

## LESSON 3 : 💰 Cost Considerations!

### 🥋 Create Another Local Database and Warehouse

In [ ]:
%%sql -r dataframe_7
use role SYSADMIN;

create database INTL_DB;

use schema INTL_DB.PUBLIC;

In [ ]:
%%sql -r dataframe_8
use role SYSADMIN;

create warehouse INTL_WH 
with 
warehouse_size = 'XSMALL' 
warehouse_type = 'STANDARD' 
auto_suspend = 600 --600 seconds/10 mins
auto_resume = TRUE;

use warehouse INTL_WH;

In [ ]:
%%sql -r dataframe_9
use warehouse INTL_WH;
create or replace table intl_db.public.INT_STDS_ORG_3166 
(iso_country_name varchar(100), 
 country_name_official varchar(200), 
 sovreignty varchar(40), 
 alpha_code_2digit varchar(2), 
 alpha_code_3digit varchar(3), 
 numeric_country_code integer,
 iso_subdivision varchar(15), 
 internet_domain_code varchar(10)
);

In [ ]:
%%sql -r dataframe_10
use role accountadmin;
use warehouse intl_wh;
create or replace file format util_db.public.PIPE_DBLQUOTE_HEADER_CR 
  type = 'CSV' --use CSV for any flat file
  compression = 'AUTO' 
  field_delimiter = '|' --pipe or vertical bar
  record_delimiter = '\r' --carriage return
  skip_header = 1  --1 header row
  field_optionally_enclosed_by = '\042'  --double quotes
  trim_space = FALSE;

### 🎯 Load the ISO Table Using Your File Forma

In [ ]:
%%sql -r dataframe_11
show stages in account; 

### 🎯 Chargez la table ISO en utilisant votre format de fichier

In [ ]:
%%sql -r dataframe_12
use role sysadmin;
use warehouse intl_wh;

CREATE OR REPLACE STAGE util_db.public.aws_s3_bucket url = 's3://uni-cmcw';

copy into intl_db.public.INT_STDS_ORG_3166
from @util_db.public.aws_s3_bucket
files = ( 'ISO_Countries_UTF8_pipe.csv')
file_format = ( format_name='util_db.public.PIPE_DBLQUOTE_HEADER_CR');

### 📓 Learning about Code Checks Using Metadata

In [ ]:
%%sql -r dataframe_13
select count(*) as found, '249' as expected 
from INTL_DB.PUBLIC.INT_STDS_ORG_3166; 

In [ ]:
%%sql -r dataframe_14
select count(*) as OBJECTS_FOUND
from INTL_DB.INFORMATION_SCHEMA.TABLES 
where table_schema='PUBLIC'
and table_name= 'INT_STDS_ORG_3166';

In [ ]:
%%sql -r dataframe_15
select row_count
from INTL_DB.INFORMATION_SCHEMA.TABLES 
where table_schema='PUBLIC'
and table_name= 'INT_STDS_ORG_3166';

### 🥋 Join Shared Data with Local Data


In [ ]:
%%sql -r dataframe_16
select  
     iso_country_name
    ,country_name_official,alpha_code_2digit
    ,r_name as region
from INTL_DB.PUBLIC.INT_STDS_ORG_3166 i
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION n
on upper(i.iso_country_name)= n.n_name
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION r
on n_regionkey = r_regionkey;

In [ ]:
%%sql -r dataframe_17
create view intl_db.public.NATIONS_SAMPLE_PLUS_ISO 
( iso_country_name
  ,country_name_official
  ,alpha_code_2digit
  ,region) AS (
  select  
     iso_country_name
    ,country_name_official,alpha_code_2digit
    ,r_name as region
from INTL_DB.PUBLIC.INT_STDS_ORG_3166 i
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION n
on upper(i.iso_country_name)= n.n_name
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION r
on n_regionkey = r_regionkey)
;

In [ ]:
%%sql -r dataframe_18
select *
from intl_db.public.NATIONS_SAMPLE_PLUS_ISO;

### 🎯 Create a Few More Tables and Load Them

In [ ]:
%%sql -r dataframe_19
create table intl_db.public.CURRENCIES 
(
  currency_ID integer, 
  currency_char_code varchar(3), 
  currency_symbol varchar(4), 
  currency_digital_code varchar(3), 
  currency_digital_name varchar(30)
)
  comment = 'Information about currencies including character codes, symbols, digital codes, etc.';

In [ ]:
%%sql -r dataframe_20
create table intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE 
  (
    country_char_code varchar(3), 
    country_numeric_code integer, 
    country_name varchar(100), 
    currency_name varchar(100), 
    currency_char_code varchar(3), 
    currency_numeric_code integer
  ) 
  comment = 'Mapping table currencies to countries';

In [ ]:
%%sql -r dataframe_21
 create file format util_db.public.CSV_COMMA_LF_HEADER
  type = 'CSV' 
  field_delimiter = ',' 
  record_delimiter = '\n' -- the n represents a Line Feed character
  skip_header = 1 
;

In [ ]:
%%sql -r dataframe_22
COPY INTO intl_db.public.currencies
from @UTIL_DB.PUBLIC.MY_INTERNAL_STAGE
files = ('currencies.xls')
file_format = (format_name='util_db.public.CSV_COMMA_LF_HEADER');



In [ ]:
%%sql -r dataframe_23

COPY INTO intl_db.public.country_code_to_currency_code
from @UTIL_DB.PUBLIC.MY_INTERNAL_STAGE
files = ('country_code_to_currency_code.xls')
file_format = (format_name='util_db.public.CSV_COMMA_LF_HEADER');

### 🎯 Create A View

In [ ]:
%%sql -r dataframe_24
CREATE OR REPLACE VIEW intl_db.public.simple_currency AS (
    SELECT 
    country_char_code as CTY_CODE,
    currency_char_code as CURL_CODE
    FROM country_code_to_currency_code
    
)

In [ ]:
%%sql -r dataframe_25
SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME();

In [ ]:
%%sql -r dataframe_26
USE ROLE ORGADMIN;
SELECT SYSTEM$ENABLE_GLOBAL_DATA_SHARING_FOR_ACCOUNT('DIB74717');

In [ ]:
%%sql -r dataframe_27
select current_account();

In [ ]:
%%sql -r dataframe_28
SELECT SYSTEM$IS_GLOBAL_DATA_SHARING_ENABLED_FOR_ACCOUNT('DIB74717');

In [ ]:
%%sql -r dataframe_29
alter view intl_db.public.NATIONS_SAMPLE_PLUS_ISO
set secure; 

alter view intl_db.public.SIMPLE_CURRENCY
set secure; 

## Lesson 6 - Shopping the Snowflake Marketplace

### 🥋 Lottie Looks Over the Data Share

In [ ]:
%%sql -r dataframe_30
use role sysadmin;
alter database PELMOREX_WEATHER_SOURCE_FROSTBYTE
rename to WEATHERSOURCE;

In [ ]:
%%sql -r dataframe_31
show tables in database WEATHERSOURCE;

## Lesson 7 : Caden explores the weather data

In [ ]:
%%sql -r dataframe_32
use database weathersource;
select country from onpoint_id.history_day where postal_code like '%8020%' group by country;

In [ ]:
%%sql -r dataframe_33
select * from   
onpoint_id.postal_codes where postal_code like '8020%' and country = 'US';

### 🎯 Convert Your Postal Code Query to a View

In [ ]:
%%sql -r dataframe_34
CREATE DATABASE MARKETINGS;

In [ ]:
%%sql -r dataframe_35
CREATE OR REPLACE SCHEMA MAILERS;

In [ ]:
%%sql -r dataframe_36
create or replace view marketings.mailers.DENVER_ZIPS AS (
    select distinct postal_code from   
    weathersource.onpoint_id.postal_codes where postal_code like '8020%' and country = 'US'
)

In [ ]:
%%sql -r dataframe_38
select count(*) from 
weathersource.onpoint_id.history_day as  h
inner join 
marketings.mailers.denver_zips as d
on h.postal_code = d.postal_code ;

In [ ]:
%%sql -r dataframe_39
select count(*) from 
weathersource.onpoint_id.forecast_day;

In [ ]:
%%sql -r history_day
select
    max(h.date_valid_std) as max_date, 
    min(h.date_valid_std) as min_date,
from 
weathersource.onpoint_id.history_day as  h
inner join 
marketings.mailers.denver_zips as d
on h.postal_code = d.postal_code ;

In [ ]:
%%sql -r dataframe_41
select
    max(h.date_valid_std) as max_date, 
    min(h.date_valid_std) as min_date,
from 
weathersource.onpoint_id.forecast_day as  h
inner join 
marketings.mailers.denver_zips as d
on h.postal_code = d.postal_code ;

In [ ]:
%%sql -r dataframe_42
select
    avg(h.AVG_CLOUD_COVER_TOT_PCT) as pourc_couv_nuageux_moyen, h.date_valid_std
from 
weathersource.onpoint_id.forecast_day as  h
inner join 
marketings.mailers.denver_zips as d
on h.postal_code = d.postal_code 
group by
h.date_valid_std
order by pourc_couv_nuageux_moyen asc

;

In [ ]:
%%sql -r dataframe_43
CREATE OR REPLACE VIEW weathersource.onpoint_id.forecast_denver_weather_sunny as (
select
    avg(h.AVG_CLOUD_COVER_TOT_PCT) as pourc_couv_nuageux_moyen, h.date_valid_std
from 
weathersource.onpoint_id.forecast_day as  h
inner join 
marketings.mailers.denver_zips as d
on h.postal_code = d.postal_code 
group by
h.date_valid_std
order by pourc_couv_nuageux_moyen asc
)

In [ ]:
%%sql -r dataframe_37
USE ROLE ACCOUNTADMIN;
SELECT SYSTEM$ENABLE_GLOBAL_DATA_SHARING_FOR_ACCOUNT('AUTO_DATA_UNLIMITED');